# Sandboxing spike (Colab T4)

Runs one R2E-Gym task end to end: sandbox, smoke test, model server, agent, grading.

Before you start: **Runtime → Change runtime type → T4 GPU**. Run the cells top to bottom.

## 0. Check the GPU
Confirms a T4 is attached. The model server needs it.

In [ ]:
!nvidia-smi

## 1. Install
- `mini-swe-agent`: the agent
- `datasets`: loads the tasks
- `udocker`: runs Docker images without Docker
- `vllm`: serves the model

Pip conflict warnings are expected. When this cell finishes, do **Runtime → Restart session** once.

In [ ]:
!pip -q install mini-swe-agent datasets udocker vllm
!udocker --allow-root install

After the restart, check that the new versions loaded:

In [ ]:
import torch
import vllm

print("vllm: ", vllm.__version__)
print("torch:", torch.__version__)
print("gpu:  ", torch.cuda.get_device_name(0))

## 2. Load a task
Picks the first task with a small fix: one file, at most 10 lines changed.

In [ ]:
import json
from datasets import load_dataset

dataset = load_dataset("R2E-Gym/R2E-Gym-Lite", split="train", streaming=True)

for row in dataset:
    if row["num_non_test_files"] == 1 and row["num_non_test_lines"] <= 10:
        task = row
        break

print("Repo: ", task["repo_name"])
print("Image:", task["docker_image"])
print("Files:", task["modified_files"])
print()
print(task["problem_statement"][:1500])

### 2b. Answer key
How each test should end after the correct fix. We grade against this.

In [ ]:
expected = json.loads(task["expected_output_json"])

print(len(expected), "tests")
for test_name, status in list(expected.items())[:10]:
    print(status, test_name)

### 2c. Gold fix
The real commit that fixed the bug. Used only in the smoke test. The agent never sees it.

In [ ]:
commit = json.loads(task["parsed_commit_content"])
first_file = commit["file_diffs"][0]

print("Commit keys:   ", list(commit.keys()))
print("File diff keys:", list(first_file.keys()))
print("Header:", json.dumps(first_file.get("header"), indent=1)[:600])

## 3. Sandbox
Downloads the task's image and creates a container from it.
`ROOT` is the container's folder on disk, so we can also edit its files from Python.

In [ ]:
import os
import re
import shlex
import shutil
import subprocess

IMAGE = task["docker_image"]
NAME = "r2e"


def sh(command):
    """Run a shell command on Colab and return its output."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    return (result.stdout + result.stderr).strip()


def in_box(command, workdir="/testbed"):
    """Run a shell command inside the container."""
    return sh(
        f"udocker --allow-root run --nobanner --workdir={workdir} {NAME} "
        f"/bin/bash -lc {shlex.quote(command)}"
    )


def fresh_box():
    """Delete the container and make a new one, so the code is buggy again."""
    global ROOT
    sh(f"udocker --allow-root rm {NAME}")
    sh(f"udocker --allow-root create --name={NAME} {IMAGE}")
    ROOT = sh(f"udocker --allow-root inspect -p {NAME}").splitlines()[-1]


print(sh(f"udocker --allow-root pull {IMAGE}")[-300:])
fresh_box()
print("ROOT:", ROOT)

### 3b. Look inside
Shows where the repo, the tests and the test script are.

In [ ]:
print("--- / ---")
print(in_box("ls", workdir="/"))

print("--- /testbed ---")
print(in_box("ls -a"))

print("--- test folders ---")
print(in_box("ls -d /r2e_tests /testbed/r2e_tests 2>/dev/null"))

print("--- run_tests.sh ---")
print(in_box("head -20 run_tests.sh"))

## 4. Grading tools
- `run_tests()`: copies the hidden tests into `/testbed` and runs them.
- `grade()`: compares the results with the answer key.
- `save_baseline()` and `get_patch()`: the agent's code changes as a `git diff`.

The image already has some setup changes in the repo (for example an `install.sh`). `save_baseline()` snapshots the code before the agent starts, so the patch shows only what the agent changed.

In [ ]:
TEST_CMD = "bash run_tests.sh"
STATUSES = ["PASSED", "FAILED", "ERROR", "SKIPPED", "XFAIL", "XPASS"]


def run_tests():
    return in_box(f"[ -e r2e_tests ] || cp -r /r2e_tests r2e_tests; {TEST_CMD}")


def parse_results(output):
    """Read pytest lines like 'PASSED tests/test_x.py::TestA::test_b'."""
    results = {}
    for line in output.splitlines():
        words = line.split()
        if len(words) < 2:
            continue
        if words[0] in STATUSES:
            results[words[1]] = words[0]
        elif words[1] in STATUSES:
            results[words[0]] = words[1]
    return results


def grade(output):
    """Return (number of tests that match the answer key, list of mismatches)."""
    results = parse_results(output)
    results = {name.replace("::", "."): status for name, status in results.items()}

    mismatches = []
    for test_name, wanted in expected.items():
        got = "missing"
        for name, status in results.items():
            if name.endswith(test_name):
                got = status
                break
        if got.upper() != str(wanted).upper():
            mismatches.append((test_name, wanted, got))

    matched = len(expected) - len(mismatches)
    return matched, mismatches


def summary(output):
    """Pytest's last summary line, e.g. '3 failed, 11 passed in 2.1s'."""
    lines = [l for l in output.splitlines() if re.search(r"passed|failed|error", l, re.I)]
    return lines[-1] if lines else "(no pytest summary)"


SNAPSHOT = "git add -A -- . ':!r2e_tests' ':!.venv' && git write-tree && git reset -q"


def snapshot():
    """Save the current code as a git tree and return its id. The index is reset after."""
    output = in_box(SNAPSHOT)
    return re.findall(r"\b[0-9a-f]{40}\b", output)[-1]


def save_baseline():
    global BASE_TREE
    BASE_TREE = snapshot()


def get_patch():
    return in_box(f"git diff {BASE_TREE} {snapshot()}")

## 5. Smoke test
Tests on the buggy code should **fail**. After the gold fix they should **pass**.
This proves the sandbox and grading work before any model is involved.

In [ ]:
def apply_gold_fix():
    """Overwrite the changed source files (not tests) with their fixed version."""
    changed = []
    for file_diff in commit["file_diffs"]:
        header = file_diff.get("header") or {}
        path = (header.get("file") or {}).get("path")
        new_content = file_diff.get("new_file_content")

        if path is None or new_content is None:
            print("Could not read this file diff. Keys:", list(file_diff.keys()))
            continue
        if "test" in path.lower():
            continue

        with open(os.path.join(ROOT, "testbed", path), "w") as f:
            f.write(new_content)
        changed.append(path)
    return changed


fresh_box()
buggy_output = run_tests()

print("Gold fix applied to:", apply_gold_fix())
fixed_output = run_tests()

print("BUGGY:", summary(buggy_output), "| matched", grade(buggy_output)[0], "/", len(expected))
print("FIXED:", summary(fixed_output), "| matched", grade(fixed_output)[0], "/", len(expected))

Good result: BUGGY has failures, FIXED matches all tests. If not, print the raw output:

In [ ]:
print(fixed_output[-2000:])

## 6. Model server
Starts vLLM with Qwen3-4B (8B does not fit on a T4).
Tool calling is on because the agent sends commands through a `bash` tool.
Safe to re-run: it stops any old server first.

In [ ]:
import time
import requests

MODEL = "Qwen/Qwen3-4B"
PORT = 8000
URL = f"http://localhost:{PORT}/v1"

sh("pkill -9 -f 'vllm serve'")
time.sleep(8)

server = subprocess.Popen(
    f"vllm serve {MODEL} --port {PORT} "
    f"--dtype half --max-model-len 16384 --gpu-memory-utilization 0.85 --enforce-eager "
    f"--enable-auto-tool-choice --tool-call-parser hermes "
    f"> vllm.log 2>&1",
    shell=True,
)


def server_is_up():
    try:
        return requests.get(f"{URL}/models", timeout=2).ok
    except requests.RequestException:
        return False


for second in range(0, 900, 10):
    if server_is_up() or server.poll() is not None:
        break
    time.sleep(10)

if server_is_up():
    print(f"Server up after about {second} s")
else:
    print("Server did not start. Last log lines:")
    print(open("vllm.log").read()[-3000:])

### 6b. Test the model
One short message with thinking turned off.

In [ ]:
reply = requests.post(f"{URL}/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Say hi in 3 words."}],
    "max_tokens": 20,
    "chat_template_kwargs": {"enable_thinking": False},
})

print(reply.json()["choices"][0]["message"]["content"])

## 7. Agent setup
- **Model:** Qwen3's recommended non-thinking settings. `presence_penalty` discourages repeating the same command. `SEED` makes the run repeatable. `THINKING = True` lets the model reason before each command (slower, usually plans better).
- **Environment:** every agent command runs inside the container.
- **Hidden tests:** moved out of the container while the agent works, so it cannot read or edit them.

In [ ]:
import glob
import yaml
import litellm
import minisweagent
from minisweagent.agents.default import DefaultAgent
from minisweagent.environments.local import LocalEnvironment
from minisweagent.models.litellm_model import LitellmModel

litellm.suppress_debug_info = True

config_dir = os.path.join(os.path.dirname(minisweagent.__file__), "config")
config_file = glob.glob(f"{config_dir}/**/swebench.yaml", recursive=True)[0]
config = yaml.safe_load(open(config_file))

SEED = 0
STEP_LIMIT = 30
THINKING = False
SAMPLING = {"temperature": 0.7, "top_p": 0.8, "presence_penalty": 1.5, "seed": SEED}

model = LitellmModel(
    model_name=f"hosted_vllm/{MODEL}",
    cost_tracking="ignore_errors",
    model_kwargs={
        "api_base": URL,
        **SAMPLING,
        "extra_body": {"top_k": 20, "chat_template_kwargs": {"enable_thinking": THINKING}},
    },
)

In [ ]:
class UdockerEnvironment(LocalEnvironment):
    """Same as the local environment, but every command runs inside the container."""

    def execute(self, action, *args, **kwargs):
        command = action["command"] if isinstance(action, dict) else action
        wrapped = (
            f"udocker --allow-root run --nobanner --workdir=/testbed {NAME} "
            f"/bin/bash -lc {shlex.quote(command)}"
        )
        if isinstance(action, dict):
            action = {**action, "command": wrapped}
        else:
            action = wrapped
        return super().execute(action, *args, **kwargs)

In [ ]:
HIDE_DIR = "/content/hidden_tests"
TEST_DIRS = ["r2e_tests", "testbed/r2e_tests"]


def hidden_path(test_dir):
    return os.path.join(HIDE_DIR, test_dir.replace("/", "__"))


def hide_tests():
    os.makedirs(HIDE_DIR, exist_ok=True)
    for test_dir in TEST_DIRS:
        inside = os.path.join(ROOT, test_dir)
        if os.path.isdir(inside) and not os.path.islink(inside):
            shutil.rmtree(hidden_path(test_dir), ignore_errors=True)
            shutil.move(inside, hidden_path(test_dir))
            print("Hid", test_dir)


def restore_tests():
    for test_dir in TEST_DIRS:
        if os.path.exists(hidden_path(test_dir)):
            shutil.move(hidden_path(test_dir), os.path.join(ROOT, test_dir))
            print("Restored", test_dir)

## 8. Run the agent
Fresh container, tests hidden, up to 30 steps. Each step takes about 10 to 30 seconds.
The tests are restored even if the run crashes.

In [ ]:
fresh_box()
save_baseline()
hide_tests()

agent_config = {**config.get("agent", {}), "step_limit": STEP_LIMIT}
agent = DefaultAgent(model, UdockerEnvironment(), **agent_config)

try:
    result = agent.run(task["problem_statement"])
    print(str(result)[:2000])
finally:
    restore_tests()

### 8b. Look at the last steps
The model's last commands and the container's replies.

In [ ]:
for message in agent.messages[-6:]:
    print(message.get("role"), "|", str(message.get("content"))[:300])
    print("-" * 60)

## 9. Grade
Saves the agent's patch first, then runs the tests on its code.

In [ ]:
patch = get_patch()
agent_output = run_tests()
matched, mismatches = grade(agent_output)

print(summary(agent_output))
print("Matched", matched, "/", len(expected))
print("RESOLVED" if not mismatches else f"NOT RESOLVED ({len(mismatches)} mismatches)")

## 10. Save the trajectory
One JSON file per run: settings, result, patch and every message.

In [ ]:
import datetime
from importlib.metadata import version

last_message = agent.messages[-1]

trajectory = {
    "task_id": task["commit_hash"],
    "repo": task["repo_name"],
    "docker_image": IMAGE,
    "condition": "unguided",
    "model": MODEL,
    "sampling": SAMPLING,
    "thinking": THINKING,
    "step_limit": STEP_LIMIT,
    "versions": {name: version(name) for name in ["mini-swe-agent", "vllm", "udocker"]},
    "finished_at": datetime.datetime.now().isoformat(),
    "exit_status": last_message.get("content") if last_message.get("role") == "exit" else None,
    "steps": sum(1 for m in agent.messages if m.get("role") == "assistant"),
    "resolved": not mismatches,
    "tests_matched": f"{matched}/{len(expected)}",
    "test_summary": summary(agent_output),
    "patch": patch,
    "messages": agent.messages,
}

OUT_FILE = f"traj_{task['commit_hash'][:8]}_seed{SEED}.json"
with open(OUT_FILE, "w") as f:
    json.dump(trajectory, f, indent=1, default=str)

print("Saved", OUT_FILE, "with", trajectory["steps"], "steps")

In [ ]:
from google.colab import files

files.download(OUT_FILE)

## Checklist
- [ ] 5: buggy fails, gold fix passes
- [ ] 6: model answers
- [ ] 8: agent runs
- [ ] 10: trajectory saved with patch and test result

On Unity, swap udocker for Apptainer and Qwen3-4B for Qwen3-8B. Everything else stays.